CartPole 环境本身的终止条件：杆偏离竖直方向约 12° 就判定失败

In [2]:
# pip install "gymnasium[classic-control]" torch numpy matplotlib

import random
from collections import deque

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim),
        )

    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        return (
            torch.tensor(np.array(states), dtype=torch.float32),
            torch.tensor(actions, dtype=torch.long).unsqueeze(1),
            torch.tensor(rewards, dtype=torch.float32).unsqueeze(1),
            torch.tensor(np.array(next_states), dtype=torch.float32),
            torch.tensor(dones, dtype=torch.float32).unsqueeze(1),
        )

    def __len__(self):
        return len(self.buffer)


env = gym.make("CartPole-v1")

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

q_net = QNetwork(state_dim, action_dim)
target_net = QNetwork(state_dim, action_dim)
target_net.load_state_dict(q_net.state_dict())

optimizer = torch.optim.Adam(q_net.parameters(), lr=1e-3)
buffer = ReplayBuffer()

gamma = 0.99
batch_size = 64
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995
episodes = 500
target_update_freq = 20


for episode in range(episodes):
    state, _ = env.reset()
    total_reward = 0 # Reward = 存活步数

    done = False

    while not done:
        # epsilon-greedy 选择动作
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
                q_values = q_net(state_tensor)
                action = q_values.argmax(dim=1).item()

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        buffer.push(state, action, reward, next_state, done)

        state = next_state
        total_reward += reward

        if len(buffer) >= batch_size:
            states, actions, rewards, next_states, dones = buffer.sample(batch_size)

            # 当前 Q(s,a)
            q_values = q_net(states).gather(1, actions)

            # 目标值：r + gamma * max Q_target(s', a')
            with torch.no_grad():
                next_q_values = target_net(next_states).max(dim=1, keepdim=True)[0]
                # 核心就这句: 当前目标价值=当前奖励+未来最大价值
                target = rewards + gamma * next_q_values * (1 - dones)

            loss = F.mse_loss(q_values, target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if episode % target_update_freq == 0:
        target_net.load_state_dict(q_net.state_dict())

    if episode % 10 == 0:
        print(f"Episode {episode}, Reward: {total_reward}, Epsilon: {epsilon:.3f}")


env.close()

Episode 0, Reward: 26.0, Epsilon: 0.995
Episode 10, Reward: 58.0, Epsilon: 0.946
Episode 20, Reward: 12.0, Epsilon: 0.900
Episode 30, Reward: 39.0, Epsilon: 0.856
Episode 40, Reward: 66.0, Epsilon: 0.814
Episode 50, Reward: 19.0, Epsilon: 0.774
Episode 60, Reward: 17.0, Epsilon: 0.737
Episode 70, Reward: 71.0, Epsilon: 0.701
Episode 80, Reward: 21.0, Epsilon: 0.666
Episode 90, Reward: 37.0, Epsilon: 0.634
Episode 100, Reward: 33.0, Epsilon: 0.603
Episode 110, Reward: 14.0, Epsilon: 0.573
Episode 120, Reward: 37.0, Epsilon: 0.545
Episode 130, Reward: 42.0, Epsilon: 0.519
Episode 140, Reward: 23.0, Epsilon: 0.493
Episode 150, Reward: 36.0, Epsilon: 0.469
Episode 160, Reward: 136.0, Epsilon: 0.446
Episode 170, Reward: 53.0, Epsilon: 0.424
Episode 180, Reward: 182.0, Epsilon: 0.404
Episode 190, Reward: 207.0, Epsilon: 0.384
Episode 200, Reward: 266.0, Epsilon: 0.365
Episode 210, Reward: 203.0, Epsilon: 0.347
Episode 220, Reward: 57.0, Epsilon: 0.330
Episode 230, Reward: 194.0, Epsilon: 0.3

## 可视化训练结果

训练时不渲染画面，可以避免绘图拖慢训练。训练结束后，再创建一个支持 `rgb_array` 的环境，让模型完整玩一局，并把每一步的画面组合成 Notebook 内的动画。

这里不再使用 epsilon-greedy，而是始终选择 Q 值最大的动作，用于观察训练后的模型实际学得怎么样。

In [3]:
import re

import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display


visual_env = gym.make("CartPole-v1", render_mode="rgb_array")
state, _ = visual_env.reset()
frames = [visual_env.render()]
total_reward = 0
done = False

q_net.eval()

while not done:
    # 可视化时不再随机探索，直接选择当前 Q 值最大的动作
    with torch.inference_mode():
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        action = q_net(state_tensor).argmax(dim=1).item()

    state, reward, terminated, truncated, _ = visual_env.step(action)
    done = terminated or truncated
    total_reward += reward
    frames.append(visual_env.render())

visual_env.close()

fig, ax = plt.subplots(figsize=(6, 4))
ax.axis("off")
image = ax.imshow(frames[0])


def update(frame):
    image.set_data(frame)
    return (image,)


cartpole_animation = animation.FuncAnimation(
    fig,
    update,
    frames=frames,
    interval=20,
    blit=True,
)
plt.close(fig)

# Matplotlib 的 JS 播放器默认停在第一帧，这里在初始化后主动开始播放
animation_html = cartpole_animation.to_jshtml(default_mode="loop")
animation_html = re.sub(
    r"(?P<name>anim[0-9a-f]+) = new Animation\((?P<args>.*?loop_select_id)\);",
    lambda match: f"{match.group(0)}\n        {match.group('name')}.play_animation();",
    animation_html,
    count=1,
    flags=re.DOTALL,
)

print(f"可视化回合 Reward: {total_reward}")
display(HTML(animation_html))

可视化回合 Reward: 139.0


## 失败示例：始终向左推

下面故意使用一个错误策略：无论小车和杆处于什么状态，动作始终选择 `0`，也就是一直向左推。由于它不会根据杆的倾斜方向调整动作，杆很快就会倒下，Reward 也会明显低于训练后的 DQN。

In [4]:
failed_env = gym.make("CartPole-v1", render_mode="rgb_array")
state, _ = failed_env.reset(seed=0)
failed_frames = [failed_env.render()]
failed_reward = 0
done = False

while not done:
    action = 0  # 错误策略：不观察状态，始终向左推
    state, reward, terminated, truncated, _ = failed_env.step(action)
    done = terminated or truncated
    failed_reward += reward
    failed_frames.append(failed_env.render())

failed_env.close()

fig, ax = plt.subplots(figsize=(6, 4))
ax.axis("off")
image = ax.imshow(failed_frames[0])


def update_failed(frame):
    image.set_data(frame)
    return (image,)


failed_animation = animation.FuncAnimation(
    fig,
    update_failed,
    frames=failed_frames,
    interval=60,
    blit=True,
)
plt.close(fig)

failed_animation_html = failed_animation.to_jshtml(default_mode="loop")
failed_animation_html = re.sub(
    r"(?P<name>anim[0-9a-f]+) = new Animation\((?P<args>.*?loop_select_id)\);",
    lambda match: f"{match.group(0)}\n        {match.group('name')}.play_animation();",
    failed_animation_html,
    count=1,
    flags=re.DOTALL,
)

print(f"失败策略 Reward: {failed_reward}")
display(HTML(failed_animation_html))

失败策略 Reward: 11.0
